#Airbnb ELT Pipeline 

##Bronze Layer

Schema Creation

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze;

Volume Creation

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.airbnb.raw_file;

Read Data from CSV File

In [0]:
df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .csv("/Volumes/workspace/airbnb/raw_file/listingsAmsterdam.csv")
)

display(df_raw.limit(5))

id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
27886,"Romantic, stylish B&B houseboat in canal district",97647,Flip,null,Centrum-West,52.38761,4.89188,Private room,132,3,311,2025-09-07,1.87,1,17,33,0363 974D 4986 7411 88D8
28871,Comfortable double room,124245,Edwin,null,Centrum-West,52.36775,4.89092,Private room,89,2,732,2025-09-07,3.99,2,126,93,0363 607B EA74 0BD8 2F6F
29051,Comfortable single / double room,124245,Edwin,null,Centrum-Oost,52.36584,4.89111,Private room,61,2,849,2025-09-08,4.81,2,95,86,0363 607B EA74 0BD8 2F6F
44391,Quiet 2-bedroom Amsterdam city centre apartment,194779,Jan,null,Centrum-Oost,52.37168,4.91471,Entire home/apt,null,3,42,2022-08-20,0.23,1,0,0,0363 E76E F06A C1DD 172C
48373,Cozy family home in Amsterdam South,220434,Vesna & Misha,null,Buitenveldert - Zuidas,52.327807756778164,4.87680005722526,Entire home/apt,null,3,5,2024-04-28,0.19,1,0,0,0363 4A2B A6AD 0196 F684


Ingestion Date Field

In [0]:
from pyspark.sql.functions import current_timestamp
df_bronze=df_raw.withColumn("ingestion_date",current_timestamp())

Column Names Standarization

In [0]:
for c in df_bronze.columns:
    new_c = c.strip().lower().replace(" ", "_").replace("$", "").replace("(", "").replace(")", "")
    print(f"Original column: {c}  →  New column: {new_c}")
    df_bronze = df_bronze.withColumnRenamed(c, new_c)



Original column: id  →  New column: id
Original column: name  →  New column: name
Original column: host_id  →  New column: host_id
Original column: host_name  →  New column: host_name
Original column: neighbourhood_group  →  New column: neighbourhood_group
Original column: neighbourhood  →  New column: neighbourhood
Original column: latitude  →  New column: latitude
Original column: longitude  →  New column: longitude
Original column: room_type  →  New column: room_type
Original column: price  →  New column: price
Original column: minimum_nights  →  New column: minimum_nights
Original column: number_of_reviews  →  New column: number_of_reviews
Original column: last_review  →  New column: last_review
Original column: reviews_per_month  →  New column: reviews_per_month
Original column: calculated_host_listings_count  →  New column: calculated_host_listings_count
Original column: availability_365  →  New column: availability_365
Original column: number_of_reviews_ltm  →  New column: numbe

In [0]:
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.bronze.airbnb_listings_amsterdam")

Validation

In [0]:
rows_df = df_raw.count()
rows_table = spark.read.table("workspace.bronze.airbnb_listings_amsterdam").count()

validation_result = "PASS" if rows_df == rows_table else "FAIL"

print(f"""
Row Count Validation:
Source rows: {rows_df}
Target rows: {rows_table}
Result: {validation_result}
""")



Row Count Validation:
Source rows: 10480
Target rows: 10480
Result: PASS

